# Baseline Evaluation — Llama 3.2-1B on Full 11,490 Test Examples

**Project:** LoRA Fine-Tuning for Text Summarization  
**Model:** Llama 3.2-1B (no fine-tuning)  
**Dataset:** CNN/DailyMail (version 3.0.0)  
**Purpose:** Establish a rigorous baseline by evaluating the raw, untrained Llama 3.2-1B model on the **complete** CNN/DailyMail test set (11,490 examples). This gives us the most statistically reliable reference point to measure how much LoRA fine-tuning actually improves performance.

---

### How this differs from `baseline_eval_1k.ipynb`
The 1k notebook evaluated on 1,000 examples using sequential (one-by-one) generation. This notebook evaluates on the **full 11,490-example test set** using **batched evaluation** (16 examples at a time). Batching makes GPU utilization jump from ~50% to ~95-100%, reducing total evaluation time from ~13 hours (sequential) to ~30 minutes.

---
### Runtime Estimate
- **GPU:** A100 (Colab Pro recommended)
- **Time:** ~30 minutes for 11,490 examples (batched evaluation, batch size=16)

## Step 1 — Install Required Libraries
We install all the necessary Python libraries:
- `transformers` — HuggingFace library to load and run pre-trained models like Llama
- `datasets` — HuggingFace library to load benchmark datasets like CNN/DailyMail
- `evaluate` — HuggingFace library for computing BLEU and ROUGE scores
- `rouge-score` — backend used by the evaluate library for ROUGE computation
- `nltk` — used internally for tokenization during BLEU calculation
- `accelerate` — helps run models efficiently on GPU

In [1]:
!pip install transformers datasets evaluate rouge-score nltk accelerate -q
print("All libraries installed successfully!")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
All libraries installed successfully!


## Step 2 — Check GPU Availability
We verify that a GPU is available and accessible via CUDA.

**What is CUDA?**  
CUDA is NVIDIA's platform that allows Python code to run on the GPU instead of CPU. Running a large language model like Llama on CPU would take days — on a GPU it takes minutes. PyTorch uses CUDA automatically when a GPU is available.

**Why does this matter more here?**  
Batched evaluation (used in this notebook) pushes GPU utilization to 95-100%. An A100 is strongly recommended. On a T4, the same evaluation would take significantly longer.

In [2]:
import torch

# Check if GPU is available
if torch.cuda.is_available():
    print(f"   GPU is available!")
    print(f"   GPU Name     : {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   No GPU found. Please enable GPU in Colab: Runtime > Change runtime type > A100 GPU")

   GPU is available!
   GPU Name     : NVIDIA A100-SXM4-40GB
   GPU Memory   : 42.4 GB


## Step 3 — Authenticate with HuggingFace
Llama 3.2 is a gated model on HuggingFace — meaning you need to:
1. Have a HuggingFace account
2. Accept Meta's license at: https://huggingface.co/meta-llama/Llama-3.2-1B
3. Generate an access token at: https://huggingface.co/settings/tokens

Paste your token when prompted below.

In [3]:
from huggingface_hub import login

# This will prompt you to enter your HuggingFace access token
# Get your token from: https://huggingface.co/settings/tokens
login()
print(" Logged in to HuggingFace!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


 Logged in to HuggingFace!


## Step 4 — Load the Full CNN/DailyMail Test Dataset (11,490 examples)
We load the **entire** CNN/DailyMail test split — all 11,490 examples. This is the complete, official test set used to produce the final reported results in our project.

**Why CNN/DailyMail?**  
It is the industry-standard benchmark dataset for news summarization research. It contains ~312k article-summary pairs from CNN and Daily Mail news articles. Using a public, well-known dataset makes our results reproducible and comparable with other research.

**Why the full 11,490 examples here?**  
The 1k notebook used a subset for quick iteration. This notebook produces the **final, statistically reliable** baseline scores. A larger test set reduces variance and gives a more accurate picture of true model performance. The 11k baseline scored higher than the 1k baseline, showing that the 1k subset happened to contain harder examples.

In [4]:
from datasets import load_dataset

print("Loading full CNN/DailyMail test set (11,490 examples)...")

# Load the complete test split — no .select() used here, we want all examples
# Version 3.0.0 is the standard version used in research
test_dataset = load_dataset("cnn_dailymail", "3.0.0", split="test")

print(f"   Loaded {len(test_dataset)} test examples")
print(f"\nDataset columns: {test_dataset.column_names}")
print(f"\n--- Sample Example ---")
print(f"Article (first 300 chars) : {test_dataset[0]['article'][:300]}")
print(f"\nReference Summary         : {test_dataset[0]['highlights']}")

Loading full CNN/DailyMail test set (11,490 examples)...


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

   Loaded 11490 test examples

Dataset columns: ['article', 'highlights', 'id']

--- Sample Example ---
Article (first 300 chars) : (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the cou

Reference Summary         : Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


## Step 5 — Load the Llama 3.2-1B Model and Tokenizer

**What is a Tokenizer?**  
A tokenizer converts raw text into numbers (tokens) that the model understands. For example, `"Hello world"` becomes `[15043, 1917]`. It also does the reverse — converting model output numbers back into readable text.

**What is AutoModelForCausalLM?**  
This loads a causal language model — a model that generates text by predicting the next token based on all previous tokens. Llama is a causal LM. "Auto" means HuggingFace automatically picks the right model architecture based on the model name.

**Why `torch_dtype=torch.float16`?**  
This loads the model in 16-bit floating point instead of 32-bit. It cuts memory usage roughly in half, allowing us to run a 1B parameter model on an A100 GPU (40GB VRAM).

**Why `padding_side='left'`?**  
For batched generation with causal language models, inputs in a batch may have different lengths and must be padded to the same length. Left-padding ensures that the actual text always ends at the same position (the right side), so all sequences in a batch align correctly when the model generates new tokens after the prompt.

In [6]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "meta-llama/Llama-3.2-1B"

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Llama does not have a dedicated padding token, so we use the end-of-sequence token instead
tokenizer.pad_token = tokenizer.eos_token

# Left-padding is required for batched generation with causal language models.
# In a batch, sequences are padded on the LEFT so that all sequences end at the
# same position — this ensures the model generates new tokens correctly for each sequence.
tokenizer.padding_side = "left"

print(f"Loading model {MODEL_NAME}...")
print("This may take a few minutes on first run (downloading ~2.5GB)...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,   # Use 16-bit precision to reduce memory usage
    device_map="auto"            # Automatically place model on GPU if available
)

# Set model to evaluation mode — disables dropout layers used only during training
model.eval()

print(f"\n   Model loaded successfully!")
print(f"   Model device : {next(model.parameters()).device}")
print(f"   Parameters   : {sum(p.numel() for p in model.parameters()):,}")

Loading tokenizer for meta-llama/Llama-3.2-1B...


config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

...

Loading model meta-llama/Llama-3.2-1B...
This may take a few minutes on first run (downloading ~2.5GB)...


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]


   Model loaded successfully!
   Model device : cuda:0
   Parameters   : 1,235,814,400


## Step 6 — Test on a Single Example
Before running the full evaluation, we test on one example to make sure the model loads correctly and generates output.

**How does text generation work here?**
1. We build a prompt: `"Summarize the following article:\n\n{article}\n\nSummary:"`
2. The tokenizer converts this prompt into token IDs
3. The model generates new tokens after `"Summary:"`
4. The tokenizer decodes the output tokens back into text
5. We extract only the part after `"Summary:"` as the generated summary

**What is `torch.no_grad()`?**  
During inference (not training), we don't need to compute gradients. `torch.no_grad()` disables gradient tracking, saving memory and speeding up inference significantly.

In [7]:
import time

# Pick the first test example
test_example = test_dataset[0]

# Build the prompt — this tells the model what task to perform
# The model will generate text that continues after "Summary:"
prompt = f"Summarize the following article:\n\n{test_example['article']}\n\nSummary:"

# Tokenize the prompt
# truncation=True   : if article is too long, cut it to max_length
# max_length=512    : maximum number of input tokens
# return_tensors='pt': return PyTorch tensors (not plain lists)
inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# Move inputs to the same device as the model (GPU)
# Model and data must be on the same device or PyTorch will throw an error
inputs = {k: v.to(model.device) for k, v in inputs.items()}

start_time = time.time()

# Generate the summary
# torch.no_grad() disables gradient computation — we are only doing inference, not training
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,                        # Generate at most 100 new tokens
        do_sample=False,                           # Greedy decoding — always pick the most probable token
        pad_token_id=tokenizer.eos_token_id        # Use EOS token for padding (Llama has no dedicated pad token)
    )

elapsed = time.time() - start_time

# Decode the output token IDs back into readable text
# skip_special_tokens=True removes tokens like <eos>, <pad> from the output
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

# Extract only the part after "Summary:" — the full output includes the input prompt
generated_summary = generated_text.split("Summary:")[-1].strip()[:200]

print("=" * 60)
print("SINGLE EXAMPLE TEST")
print("=" * 60)
print(f"Article (first 200 chars) : {test_example['article'][:200]}")
print(f"\nReference Summary         : {test_example['highlights']}")
print(f"\nGenerated Summary         : {generated_summary}")
print(f"\nGeneration Time           : {elapsed:.2f} seconds")
print("\n   Single example test passed! Proceeding to full batched evaluation...")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


SINGLE EXAMPLE TEST
Article (first 200 chars) : (CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territor

Reference Summary         : Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .

Generated Summary         : Summarize the following article:

(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alle

Generation Time           : 3.32 seconds

   Single example test passed! Proceeding to full batched evaluation...


## Step 7 — Run Batched Baseline Evaluation on 11,490 Examples

We now run the untrained Llama model on all 11,490 test examples using **batched evaluation**.

**Why batched evaluation?**  
Sequential evaluation (one example at a time) leaves the GPU mostly idle between generations, achieving only ~50-60% GPU utilization. Batched evaluation processes 16 examples simultaneously, pushing GPU utilization to 95-100%. This reduces total evaluation time from ~13 hours (sequential) to ~30 minutes — a huge speedup with identical results.

**How batching works:**
1. We group examples into batches of 16
2. All 16 prompts in a batch are tokenized together with padding (shorter sequences get left-padded to match the longest)
3. The model generates summaries for all 16 simultaneously
4. We decode all 16 outputs and extract the summary portion from each

**Why `attention_mask`?**  
When sequences in a batch have different lengths, padding tokens are added. The attention mask tells the model which tokens are real (1) and which are padding (0), so the model doesn't attend to padding positions when generating.

**What are we collecting?**
- `all_summaries` — the model's generated summaries
- `all_references` — the ground truth summaries from the dataset

We compare these two lists using BLEU and ROUGE metrics in the next step.

In [9]:
from tqdm import tqdm
import time

# Batch size: number of examples processed simultaneously
# 16 works well on an A100 with a 1B model — pushes GPU utilization to ~95-100%
# If you get an out-of-memory error, reduce this to 8
BATCH_SIZE = 16

all_summaries = []    # Will store model-generated summaries
all_references = []   # Will store ground truth reference summaries

total_examples = len(test_dataset)
num_batches = (total_examples + BATCH_SIZE - 1) // BATCH_SIZE  # ceiling division

print(f"Starting batched baseline evaluation on {total_examples:,} test examples...")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Total batches   : {num_batches}")
print(f"Estimated time  : ~30 minutes on A100 GPU")
print("=" * 60)

start_total = time.time()

# Process examples in batches
for batch_idx in tqdm(range(num_batches), desc="Evaluating batches"):

    # Calculate the start and end index for this batch
    start_idx = batch_idx * BATCH_SIZE
    end_idx   = min(start_idx + BATCH_SIZE, total_examples)  # don't go past the end

    # Extract the batch of examples from the dataset
    batch = test_dataset.select(range(start_idx, end_idx))

    # Build prompts for all examples in this batch
    prompts = [
        f"Summarize the following article:\n\n{example['article']}\n\nSummary:"
        for example in batch
    ]

    # Tokenize the entire batch at once
    # padding=True     : pad shorter sequences to match the longest in the batch
    # truncation=True  : truncate sequences longer than max_length
    # max_length=512   : maximum input tokens per example
    # return_tensors='pt': return PyTorch tensors
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512
    )

    # Move the entire batch to GPU
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate summaries for all examples in the batch simultaneously
    # torch.no_grad() disables gradient computation — inference only, not training
    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],  # tells model which tokens are real vs padding
            max_new_tokens=100,                        # generate at most 100 new tokens per example
            do_sample=False,                           # greedy decoding — always pick most probable token
            pad_token_id=tokenizer.eos_token_id        # use EOS token for padding
        )

    # Decode and extract summary for each example in the batch
    for i, output in enumerate(outputs):
        # Decode the full output (includes the input prompt)
        generated_text = tokenizer.decode(output, skip_special_tokens=True)

        # Extract only the text after "Summary:" — that is the generated summary
        generated_summary = generated_text.split("Summary:")[-1].strip()

        all_summaries.append(generated_summary)

    # Collect the reference summaries for this batch
    all_references.extend(batch["highlights"])

    # Print a progress update every 500 examples
    examples_done = end_idx
    if examples_done % 500 == 0 or examples_done == total_examples:
        elapsed = time.time() - start_total
        print(f"   [{examples_done:,}/{total_examples:,}] completed | Time elapsed: {elapsed/60:.1f} mins")

total_time = time.time() - start_total
print(f"\n   Evaluation complete!")
print(f"   Total examples evaluated : {len(all_summaries):,}")
print(f"   Total time               : {total_time/60:.1f} minutes")

Starting batched baseline evaluation on 11,490 test examples...
Batch size      : 16
Total batches   : 719
Estimated time  : ~30 minutes on A100 GPU


Evaluating batches:  17%|█▋        | 125/719 [04:54<23:24,  2.36s/it]

   [2,000/11,490] completed | Time elapsed: 4.9 mins


Evaluating batches:  35%|███▍      | 250/719 [09:49<18:24,  2.35s/it]

   [4,000/11,490] completed | Time elapsed: 9.8 mins


...

   [8,000/11,490] completed | Time elapsed: 19.6 mins


Evaluating batches:  87%|████████▋ | 625/719 [24:27<03:35,  2.29s/it]

   [10,000/11,490] completed | Time elapsed: 24.5 mins


Evaluating batches: 100%|██████████| 719/719 [28:06<00:00,  2.35s/it]

   [11,490/11,490] completed | Time elapsed: 28.1 mins

   Evaluation complete!
   Total examples evaluated : 11,490
   Total time               : 28.1 minutes


## Step 8 — Calculate BLEU and ROUGE Scores

We now compare the model's generated summaries against the reference summaries using four metrics:

| Metric | What it measures |
|--------|------------------|
| **BLEU-4** | Overlap of 4-word phrases between generated and reference summary |
| **ROUGE-1** | Overlap of individual words (unigrams) |
| **ROUGE-2** | Overlap of 2-word phrases (bigrams) |
| **ROUGE-L** | Longest common subsequence between generated and reference |

**Note:** For abstractive summarization, scores in the range 10–40 are typical. The model paraphrases rather than copying — so low scores don't mean failure, they are expected. What matters is the **improvement** after fine-tuning.


In [10]:
from evaluate import load
import numpy as np

print("Calculating BLEU and ROUGE scores...")
print("=" * 60)

# Load evaluation metrics from HuggingFace evaluate library
bleu_metric  = load("bleu")
rouge_metric = load("rouge")

# Both BLEU and ROUGE receive raw strings — the evaluate library handles tokenization internally
# predictions : list of generated summary strings
# references  : for BLEU, each reference is wrapped in a list (supports multiple references per example)
#               for ROUGE, plain list of strings is fine
bleu_result = bleu_metric.compute(
    predictions=all_summaries,
    references=[[r] for r in all_references]   # each reference wrapped in a list
)

rouge_result = rouge_metric.compute(
    predictions=all_summaries,
    references=all_references                   # plain list of strings
)

# Extract BLEU-4 score (index 3 = 4-gram precision) and scale to 0-100
bleu4  = bleu_result['precisions'][3] * 100
rouge1 = rouge_result['rouge1'] * 100
rouge2 = rouge_result['rouge2'] * 100
rougeL = rouge_result['rougeL'] * 100

print(f"\n{'='*60}")
print(f"  BASELINE RESULTS — Llama 3.2-1B | 11,490 Test Examples")
print(f"{'='*60}")
print(f"  BLEU-4   : {bleu4:.2f}")
print(f"  ROUGE-1  : {rouge1:.2f}")
print(f"  ROUGE-2  : {rouge2:.2f}")
print(f"  ROUGE-L  : {rougeL:.2f}")
print(f"{'='*60}")
print(f"\nNote: These are baseline scores with NO fine-tuning.")
print(f"Compare with LoRA fine-tuned results in the lora_full notebooks.")

Calculating BLEU and ROUGE scores...



  BASELINE RESULTS — Llama 3.2-1B | 11,490 Test Examples
  BLEU-4   : 2.02
  ROUGE-1  : 21.37
  ROUGE-2  : 10.80
  ROUGE-L  : 14.77

Note: These are baseline scores with NO fine-tuning.
Compare with LoRA fine-tuned results in the lora_full notebooks.


## Step 9 — Save Results
We save the scores to a JSON file in the shared `results/` folder so they can be loaded and compared alongside all other experiments in the project.

In [12]:
import json

# Store all results in a dictionary
baseline_results_11k = {
    "experiment"  : "Baseline — Llama 3.2-1B (no fine-tuning)",
    "model"       : "meta-llama/Llama-3.2-1B",
    "test_samples": len(all_summaries),
    "bleu_4"      : round(bleu4, 4),
    "rouge_1"     : round(rouge1, 4),
    "rouge_2"     : round(rouge2, 4),
    "rouge_L"     : round(rougeL, 4)
}

# Save to current directory (Colab's /content/ folder)
# To download: right-click the file in the Colab file browser (left sidebar) and select Download
with open("baseline_results_11k.json", "w") as f:
    json.dump(baseline_results_11k, f, indent=2)

print("   Results saved to baseline_results_11k.json")
print(json.dumps(baseline_results_11k, indent=2))

   Results saved to baseline_results_11k.json
{
  "experiment": "Baseline \u2014 Llama 3.2-1B (no fine-tuning)",
  "model": "meta-llama/Llama-3.2-1B",
  "test_samples": 11490,
  "bleu_4": 2.0184,
  "rouge_1": 21.3652,
  "rouge_2": 10.804,
  "rouge_L": 14.7681
}
